# Cell Nucleus Detection & Tracking
 

This notebook goes through the full pipeline I built for this project:
1. Fine-tune YOLOv8 to detect cell nuclei in phase-contrast images
2. Apply the model and collect bounding box coordinates
3. Convert bounding boxes to pixel centre points `(cx, cy)`
4. Track nuclei across frames using Trackpy
5. Compute and plot MSD



## Imports

In [ ]:
import os
import re
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import trackpy as tp
from ultralytics import YOLO

os.makedirs('data/predictions', exist_ok=True)
os.makedirs('results/trajectories', exist_ok=True)
os.makedirs('results/msd', exist_ok=True)

## 1. Train the model

I trainned YOLOv8  on 15 manually annotated phase-contrast images.  
The annotation was done with **Label Studio** (Image Object Detection template), exported in YOLO format.

Since the dataset is small, I added data augmentation (flips, rotations, mosaic) to help the model generalize.

Output: fine-tuned model weights at `models/cell_nucleus_detector/weights/best.pt`.

In [ ]:
model = YOLO('yolov8s.pt')

model.train(
    data='./data/annotated/data.yaml',
    epochs=100,
    imgsz=640,
    project='models',
    name='cell_nucleus_detector',
    fliplr=0.5,   # horizontal flip
    flipud=0.5,   # vertical flip
    degrees=15,   # random rotation
    mosaic=0.5,   # mosaic augmentation
)

## 2. Apply the model — collect raw bounding box coordinates

I run the trained model on every image and collect the raw YOLO output: normalized coordinates  
`(xc_norm, yc_norm, w_norm, h_norm)` — values between 0 and 1, relative to image dimensions.  



In [ ]:
IMAGES_DIR = './data/raw_images'
WEIGHTS    = './models/cell_nucleus_detector/weights/best.pt'
OUTPUT_DIR = './data/predictions'

model = YOLO(WEIGHTS)

def extract_frame(filename):
    """Get the frame index from a filename like t_08.png → 8."""
    match = re.search(r'(\d+)', filename)
    return int(match.group(1)) if match else 0

image_files = sorted([f for f in os.listdir(IMAGES_DIR) if f.endswith('.png')])
raw_records = []

for img_file in image_files:
    img_path = os.path.join(IMAGES_DIR, img_file)
    frame    = extract_frame(img_file)
    results  = model(img_path)

    # save the annotated image (with boxes drawn by YOLO)
    annotated = results[0].plot()
    cv2.imwrite(os.path.join(OUTPUT_DIR, img_file), annotated)

    # collect raw normalized coordinates for each detected box
    boxes = results[0].boxes
    img_h, img_w = results[0].orig_shape

    if boxes is not None:
        for box in boxes:
            xc_norm, yc_norm, w_norm, h_norm = box.xywhn[0].tolist()
            raw_records.append({
                'frame'      : frame,
                'filename'   : img_file,
                'img_w'      : img_w,
                'img_h'      : img_h,
                'xc_norm'    : round(xc_norm, 6),
                'yc_norm'    : round(yc_norm, 6),
                'w_norm'     : round(w_norm,  6),
                'h_norm'     : round(h_norm,  6),
                'confidence' : round(float(box.conf[0]), 4),
            })

    print(f'frame {frame:03d} — {len(boxes) if boxes else 0} nucleus/nuclei detected')

# show a sample result
sample = cv2.cvtColor(cv2.imread(os.path.join(OUTPUT_DIR, image_files[0])), cv2.COLOR_BGR2RGB)
plt.figure(figsize=(8, 6))
plt.imshow(sample)
plt.title(f'Detection result — {image_files[0]}')
plt.axis('off')
plt.show()

raw_df = pd.DataFrame(raw_records).sort_values('frame').reset_index(drop=True)
print(f'\n{len(raw_df)} detections collected across {raw_df["frame"].nunique()} frames')
raw_df.head()

## 3. Convert bounding boxes to pixel centre points

YOLO gives normalized coordinates (between 0 and 1).I need to convert them into actual pixel coordinates so I can use them for tracking, as this is the unit used by Trackpy.

The idea is to reduce each bounding box to a single point — its centre `(cx, cy)`. 
This point is what Trackpy will use to link the same nucleus across consecutive frames.

```
cx = xc_norm × image_width
cy = yc_norm × image_height
```

In [ ]:
def box_to_center(xc_norm, yc_norm, img_w, img_h):
    """
    Convert normalized YOLO bounding box center to pixel coordinates.

    Parameters
    ----------
    xc_norm : float  normalized x center (0 to 1)
    yc_norm : float  normalized y center (0 to 1)
    img_w   : int    image width in pixels
    img_h   : int    image height in pixels

    Returns
    -------
    (cx, cy) : pixel coordinates of the box centre
    """
    cx = int(xc_norm * img_w)
    cy = int(yc_norm * img_h)
    return cx, cy


raw_df[['cx', 'cy']] = raw_df.apply(
    lambda row: box_to_center(row['xc_norm'], row['yc_norm'], row['img_w'], row['img_h']),
    axis=1,
    result_type='expand'
)

detections = raw_df[['frame', 'filename', 'xc_norm', 'yc_norm', 'w_norm', 'h_norm', 'cx', 'cy', 'confidence']]
detections.to_csv('data/predictions/detections.csv', index=False)

print('centre points computed — preview:')
detections.head()

## 4. Track nuclei across frames

I used Trackpy to link the detected nuclei across frames. It was originally built for  
fluorescence microscopy but works here since we're giving it centre points directly.

Two key parameters:
- `search_range = 30 px` — how far a nucleus can move between two frames and still be linked to the same track
- `memory = 4` — how many frames a nucleus can disappear (missed detection) before the track is broken

After linking, I used `filter_stubs` to remove very short trajectories that are likely noise.

In [ ]:
# trackpy expects columns: x, y, frame
df = detections[['frame', 'cx', 'cy']].rename(columns={'cx': 'x', 'cy': 'y'})

tp.quiet()

linked = tp.link(df, search_range=30, memory=4)
print(f'trajectories before filtering: {linked["particle"].nunique()}')

tracked = tp.filter_stubs(linked, threshold=5)
print(f'trajectories after filtering:  {tracked["particle"].nunique()}')

tracked.to_csv('results/trajectories/tracked.csv', index=False)
tracked.head()

In [ ]:
# lifetime of each nucleus (how many frames it appears in)
lifetime = tracked.groupby('particle')['frame'].agg(min_frame='min', max_frame='max').reset_index()
lifetime['lifetime'] = lifetime['max_frame'] - lifetime['min_frame']
lifetime.sort_values('lifetime', ascending=False).reset_index(drop=True)

In [ ]:
# all trajectories on the same plot
fig, ax = plt.subplots(figsize=(10, 8))
palette = sns.color_palette('tab10', n_colors=tracked['particle'].nunique())

for i, (pid, group) in enumerate(tracked.groupby('particle')):
    ax.plot(group['x'], group['y'], marker='o', markersize=3,
            linewidth=1.5, label=f'nucleus {pid}', color=palette[i % len(palette)])

ax.set_xlabel('x [px]')
ax.set_ylabel('y [px]')
ax.set_title('Trajectories of all tracked nuclei')
ax.invert_yaxis()  # image y-axis is top-down
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('results/trajectories/all_trajectories.png', dpi=150)
plt.show()

In [ ]:
# individual trajectory — change nucleus_id to look at a specific one
nucleus_id = tracked['particle'].iloc[0]
group = tracked[tracked['particle'] == nucleus_id]

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(group['x'], group['y'], marker='o', markersize=5, linewidth=2, color='royalblue')
ax.scatter(group['x'].iloc[0],  group['y'].iloc[0],  color='green', zorder=5, label='start', s=80)
ax.scatter(group['x'].iloc[-1], group['y'].iloc[-1], color='red',   zorder=5, label='end',   s=80)
ax.set_xlabel('x [px]')
ax.set_ylabel('y [px]')
ax.set_title(f'Trajectory of nucleus {nucleus_id}')
ax.invert_yaxis()
ax.legend()
plt.tight_layout()
plt.show()

## 5. MSD analysis

The Mean Squared Displacement (MSD) tells us how far nuclei move on average over time.  
Looking at the slope on a log-log scale helps identify the type of motion:

- slope ≈ 1 → free / Brownian diffusion
- slope > 1.2 → directed movement
- slope < 0.8 → confined movement

You need to set `FPS` and `MPP` according to your microscope's acquisition parameters.

In [ ]:
FPS = 1.0   # frames per second — adjust to your setup
MPP = 0.5   # microns per pixel — adjust to your microscope

imsd = tp.imsd(tracked, mpp=MPP, fps=FPS)  # one MSD curve per nucleus
emsd = tp.emsd(tracked, mpp=MPP, fps=FPS)  # ensemble average

imsd.to_csv('results/msd/imsd.csv')
emsd.to_csv('results/msd/emsd.csv', header=['msd'])

fig, ax = plt.subplots(figsize=(9, 6))
colors = plt.cm.tab10.colors

for i, col in enumerate(imsd.columns):
    ax.plot(imsd.index, imsd[col], linestyle='--', alpha=0.75,
            color=colors[i % len(colors)], label=f'IMSD nucleus {col}')

ax.plot(emsd.index, emsd, color='red', linewidth=2.5,
        marker='o', markersize=5, label='ensemble MSD', zorder=5)

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('lag time t (s)', fontsize=12)
ax.set_ylabel(r'$\langle \Delta r^2(t) \rangle$ ($\mu m^2$)', fontsize=12)
ax.set_title('MSD — individual and ensemble', fontsize=13)
ax.legend(fontsize=8, loc='upper left')
ax.grid(True, which='both', linestyle=':', alpha=0.5)
plt.tight_layout()
plt.savefig('results/msd/msd_comparison.png', dpi=150)
plt.show()

In [ ]:
# quick motion type estimate from the slope
half = len(emsd) // 2
slope, _ = np.polyfit(np.log10(emsd.index[:half]), np.log10(emsd.values[:half]), 1)
print(f'MSD slope (alpha) ≈ {slope:.2f}')
if slope < 0.8:
    print('→ confined / subdiffusive motion')
elif slope > 1.2:
    print('→ directed / superdiffusive motion')
else:
    print('→ free / Brownian diffusion')